# 07 — Fetch Forest Compartment Details

Fetches detailed forestry attributes (species, age, volume, increment)
for a sample of compartments via the portal REST API.

**Endpoint:** `register.metsad.ee/portaal/api/rest/eraldis/detail/{id}`
(publicly accessible, no auth required)

**Configure:**
- `SAMPLE_FRAC` or `SAMPLE_N` — how many compartments to fetch
- `N_WORKERS` — parallel requests (10 is safe, 20 is faster)
- `PROGRESS_EVERY` — print timing every N completions

In [1]:
import sys
sys.path.insert(0, "../src")

import geopandas as gpd
import pandas as pd
from pathlib import Path

from carbon_dataset.forest_registry_details import fetch_details_parallel

## Configuration

In [7]:
# --- CONFIGURE THESE ---
SAMPLE_FRAC = 0.2        # 1% = ~2,200 compartments
SAMPLE_N = None           # Set to exact number to override SAMPLE_FRAC (e.g. 500)
N_WORKERS = 10            # Parallel requests (10 = safe, 20 = faster)
PROGRESS_EVERY = 200      # Print progress every N completions

# Paths
INPUT_FILE = Path("../data/raw/forest_registry/laane_eraldised.gpkg")
OUTPUT_DIR = Path("../data/raw/forest_registry")
OUTPUT_FILE = OUTPUT_DIR / "laane_details.parquet"

## Load compartment IDs

In [8]:
gdf = gpd.read_file(INPUT_FILE)
print(f"Loaded: {len(gdf):,} compartments")

# Sample
if SAMPLE_N is not None:
    sample_ids = gdf["eraldis_id"].sample(n=min(SAMPLE_N, len(gdf)), random_state=42).tolist()
else:
    sample_ids = gdf["eraldis_id"].sample(frac=SAMPLE_FRAC, random_state=42).tolist()

n_to_fetch = len(sample_ids)
est_time = n_to_fetch * 0.2 / N_WORKERS
print(f"\nWill fetch: {n_to_fetch:,} compartments")
print(f"Estimated time: {est_time:.0f} seconds (at {N_WORKERS} workers)")

Loaded: 218,133 compartments

Will fetch: 43,627 compartments
Estimated time: 873 seconds (at 10 workers)


## Fetch details

In [9]:
details_df = fetch_details_parallel(
    sample_ids,
    n_workers=N_WORKERS,
    progress_every=PROGRESS_EVERY,
)

print(f"\nResult: {len(details_df)} rows, {len(details_df.columns)} columns")

Fetching details for 43,627 compartments (10 parallel workers)
     200/43,627 (0%) - 93 req/s - elapsed: 2s - ETA: 466s
     400/43,627 (1%) - 154 req/s - elapsed: 3s - ETA: 281s
     600/43,627 (1%) - 195 req/s - elapsed: 3s - ETA: 221s
     800/43,627 (2%) - 227 req/s - elapsed: 4s - ETA: 189s
   1,000/43,627 (2%) - 251 req/s - elapsed: 4s - ETA: 170s
   1,200/43,627 (3%) - 271 req/s - elapsed: 4s - ETA: 157s
   1,400/43,627 (3%) - 287 req/s - elapsed: 5s - ETA: 147s
   1,600/43,627 (4%) - 300 req/s - elapsed: 5s - ETA: 140s
   1,800/43,627 (4%) - 309 req/s - elapsed: 6s - ETA: 135s
   2,000/43,627 (5%) - 318 req/s - elapsed: 6s - ETA: 131s
   2,200/43,627 (5%) - 326 req/s - elapsed: 7s - ETA: 127s
   2,400/43,627 (6%) - 334 req/s - elapsed: 7s - ETA: 124s
   2,600/43,627 (6%) - 340 req/s - elapsed: 8s - ETA: 121s
   2,800/43,627 (6%) - 345 req/s - elapsed: 8s - ETA: 118s
   3,000/43,627 (7%) - 350 req/s - elapsed: 9s - ETA: 116s
   3,200/43,627 (7%) - 352 req/s - elapsed: 9s - ETA:

## Explore results

In [10]:
if len(details_df) > 0:
    print("Columns:")
    for col in sorted(details_df.columns):
        fill = details_df[col].notna().mean() * 100
        print(f"  {col}: {fill:.0f}% filled")

Columns:
  alaGeoJson: 100% filled
  alamYksuseNimi: 100% filled
  arenguklKood: 100% filled
  boniteediKood: 100% filled
  eraldis_id: 100% filled
  eraldiseNr: 100% filled
  esmaneDokId: 66% filled
  etapp: 100% filled
  id: 100% filled
  inventKp: 100% filled
  juurdekasv: 52% filled
  juurdekasvTm: 52% filled
  kasvukohaKood: 100% filled
  katastriNr: 100% filled
  keskmDiameeter: 51% filled
  keskmRaievanus: 48% filled
  keskmVanus: 73% filled
  korgus: 96% filled
  korraldaja: 100% filled
  korraldajaId: 66% filled
  kuivendatud: 100% filled
  kvartaliNr: 44% filled
  layer1_age: 98% filled
  layer1_origin: 98% filled
  layer1_share: 98% filled
  layer1_species: 98% filled
  layer1_volume: 60% filled
  liitusA: 22% filled
  maakond: 100% filled
  mkood: 66% filled
  omandivormiKood: 66% filled
  omandivormiKp: 61% filled
  peapuuliik: 100% filled
  pindala: 100% filled
  puudeArv: 12% filled
  puudeArvJ: 4% filled
  puudeArvY: 11% filled
  regKp: 100% filled
  rpindala1: 52% fill

In [11]:
if len(details_df) > 0:
    print("\nKey forestry stats:")
    stats_cols = {
        "peapuuliik": "Main species",
        "keskmVanus": "Mean age",
        "boniteediKood": "Site class",
        "juurdekasv": "Increment (m³/ha/yr)",
        "tagavara1Ha": "Volume layer1 (m³/ha)",
        "pindala": "Area (ha)",
        "kuivendatud": "Drained",
        "kasvukohaKood": "Site type",
    }
    for col, label in stats_cols.items():
        if col in details_df.columns:
            s = details_df[col]
            if s.dtype in ["float64", "int64"]:
                print(f"  {label}: mean={s.mean():.1f}, median={s.median():.1f}, "
                      f"min={s.min()}, max={s.max()}")
            else:
                top = s.value_counts().head(5).to_dict()
                print(f"  {label}: {top}")


Key forestry stats:
  Main species: {'KS': 14473, 'MA': 12582, 'KU': 5600, 'LV': 4550, 'HB': 3275}
  Mean age: mean=72.6, median=73.0, min=-10.0, max=278.0
  Site class: {'2': 18090, '3': 12915, '1': 5871, '4': 4349, '5': 1341}
  Increment (m³/ha/yr): mean=4.9, median=4.5, min=0.0, max=18.7
  Volume layer1 (m³/ha): mean=148.0, median=150.0, min=0.0, max=590.0
  Area (ha): mean=1.2, median=0.8, min=0.04, max=37.51
  Drained: {False: 29554, True: 14073}
  Site type: {'AN': 11000, 'SL': 6780, 'TA': 5101, 'TR': 2693, 'KL': 2441}


## Save

In [12]:
if len(details_df) > 0:
    details_df.to_parquet(OUTPUT_FILE, index=False)
    print(f"Saved {len(details_df):,} rows to {OUTPUT_FILE}")
    print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB")
else:
    print("No data to save!")

Saved 43,627 rows to ..\data\raw\forest_registry\laane_details.parquet
File size: 17463 KB


## Next steps

With this data you can:
1. Compute observed tCO2/ha/yr from `juurdekasv` × expansion factors
2. Train a predictor: (species, age, site_class, drained, site_type) → tCO2/ha/yr
3. Spatial join with your 1km grid to get per-cell growth estimates